# 🚀 ASTRONOVA AI Video Generator

Generate faceless videos, LoFi videos, sleep videos, and viral shorts using AI.

**Run each cell in order (Shift+Enter)**

## 1️⃣ Setup - Run this first!

In [ ]:
# Install dependencies
!pip install -q httpx edge-tts moviepy Pillow numpy
print("✅ Dependencies installed!")

In [ ]:
# Enter your API keys here (FREE to get)
# Groq: https://console.groq.com/keys
# Pexels: https://www.pexels.com/api/

GROQ_API_KEY = "YOUR_GROQ_API_KEY"  # @param {type:"string"}
PEXELS_API_KEY = "YOUR_PEXELS_API_KEY"  # @param {type:"string"}

import os
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['PEXELS_API_KEY'] = PEXELS_API_KEY
print("✅ API keys configured!")

In [ ]:
# Core imports and functions
import asyncio
import json
import random
from pathlib import Path
import httpx
import edge_tts
from moviepy import ImageClip, AudioFileClip, concatenate_videoclips, CompositeVideoClip, ColorClip, TextClip
from PIL import Image, ImageDraw, ImageFont
import numpy as np

TTS_VOICE = "en-US-AndrewNeural"

async def generate_script(topic: str, scenes: int = 10) -> list:
    """Generate video script using Groq (FREE)"""
    prompt = f"""Generate a {scenes}-scene video script about: {topic}
Return JSON only: {{"scenes": [{{"narration": "text", "image_search": "keywords"}}]}}"""

    async with httpx.AsyncClient() as client:
        r = await client.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
            json={"model": "llama-3.3-70b-versatile", "messages": [{"role": "user", "content": prompt}]},
            timeout=60
        )
        content = r.json()["choices"][0]["message"]["content"]
        start = content.find("{")
        end = content.rfind("}") + 1
        return json.loads(content[start:end])["scenes"]

async def get_pexels_image(query: str, index: int, output_dir: Path) -> str:
    """Fetch image from Pexels (FREE)"""
    async with httpx.AsyncClient() as client:
        r = await client.get(
            "https://api.pexels.com/v1/search",
            headers={"Authorization": PEXELS_API_KEY},
            params={"query": query, "per_page": 5, "orientation": "landscape"}
        )
        photos = r.json().get("photos", [])
        if photos:
            url = random.choice(photos)["src"]["large2x"]
            img = await client.get(url)
            path = output_dir / f"img_{index:03d}.jpg"
            path.write_bytes(img.content)
            return str(path)
    return ""

async def generate_tts(text: str, index: int, output_dir: Path) -> str:
    """Generate voiceover using Edge TTS (FREE)"""
    path = output_dir / f"audio_{index:03d}.mp3"
    communicate = edge_tts.Communicate(text, TTS_VOICE)
    await communicate.save(str(path))
    return str(path)

def assemble_video(scenes_data: list, output_path: str):
    """Combine images and audio into video"""
    clips = []
    for img_path, audio_path in scenes_data:
        audio = AudioFileClip(audio_path)
        img = ImageClip(img_path).with_duration(audio.duration)
        img = img.with_position('center').with_audio(audio)
        clips.append(img)

    final = concatenate_videoclips(clips, method="compose")
    final.write_videofile(output_path, fps=24, codec="libx264", audio_codec="aac")
    return output_path

print("✅ Functions loaded!")

## 2️⃣ Generate Long-Form Video

In [ ]:
# @title Generate Video { run: "auto" }
TOPIC = "5 Mind-Blowing Facts About the Ocean"  # @param {type:"string"}
NUM_SCENES = 5  # @param {type:"slider", min:3, max:15, step:1}

async def generate_video(topic: str, num_scenes: int = 10):
    output_dir = Path("./outputs")
    output_dir.mkdir(exist_ok=True)
    temp_dir = output_dir / "temp"
    temp_dir.mkdir(exist_ok=True)

    print(f"🎬 Generating video about: {topic}")

    print("📝 Generating script...")
    scenes = await generate_script(topic, num_scenes)
    print(f"   Generated {len(scenes)} scenes")

    print("🖼️ Fetching images and generating voiceovers...")
    scene_files = []
    for i, scene in enumerate(scenes):
        img = await get_pexels_image(scene["image_search"], i, temp_dir)
        audio = await generate_tts(scene["narration"], i, temp_dir)
        if img and audio:
            scene_files.append((img, audio))
        print(f"   Scene {i+1}/{len(scenes)} complete")

    print("🎥 Assembling video...")
    output_path = str(output_dir / f"{topic.replace(' ', '_')[:30]}.mp4")
    assemble_video(scene_files, output_path)

    print(f"✅ Video saved: {output_path}")
    return output_path

# Run generation
video_path = await generate_video(TOPIC, NUM_SCENES)

In [ ]:
# Play the generated video
from IPython.display import Video
Video(video_path, embed=True, width=640)

In [ ]:
# Download the video
from google.colab import files
files.download(video_path)

## 3️⃣ Generate Viral Short

In [ ]:
# @title Generate Viral Short { run: "auto" }
SHORT_TOPIC = "productivity hacks"  # @param {type:"string"}
FORMAT_TYPE = "tips"  # @param ["facts", "tips", "mistakes", "ranking"]

VIRAL_FORMATS = {
    "facts": {
        "prompt": "Generate 5 mind-blowing facts about {topic}. Each fact should be surprising. Format as JSON: {{\"facts\": [\"fact1\", \"fact2\", ...]}}",
        "hook": "You won't believe this...",
        "cta": "Follow for more!"
    },
    "tips": {
        "prompt": "Generate 5 quick actionable tips about {topic}. Format as JSON: {{\"tips\": [\"tip1\", \"tip2\", ...]}}",
        "hook": "This changed everything...",
        "cta": "Save this for later!"
    },
    "mistakes": {
        "prompt": "Generate 5 common mistakes people make with {topic}. Format as JSON: {{\"mistakes\": [\"mistake1\", ...]}}",
        "hook": "Stop making these mistakes!",
        "cta": "Share with someone who needs this!"
    },
    "ranking": {
        "prompt": "Generate a ranking of 5 items related to {topic}. Format as JSON: {{\"items\": [\"item1\", ...]}}",
        "hook": "Ranking time...",
        "cta": "What's your #1?"
    }
}

async def get_vertical_image(query: str, output_path: Path) -> str:
    async with httpx.AsyncClient() as client:
        r = await client.get(
            "https://api.pexels.com/v1/search",
            headers={"Authorization": PEXELS_API_KEY},
            params={"query": query, "per_page": 10, "orientation": "portrait"}
        )
        photos = r.json().get("photos", [])
        if photos:
            url = random.choice(photos)["src"]["large2x"]
            img = await client.get(url)
            output_path.write_bytes(img.content)
            return str(output_path)
    return ""

async def generate_short(topic: str, format_type: str = "facts"):
    output_dir = Path("./outputs/shorts")
    output_dir.mkdir(parents=True, exist_ok=True)
    temp_dir = output_dir / "temp"
    temp_dir.mkdir(exist_ok=True)

    print(f"🎬 Creating viral short: {topic}")

    format_config = VIRAL_FORMATS[format_type]
    prompt = format_config["prompt"].format(topic=topic)

    print("📝 Generating content...")
    async with httpx.AsyncClient() as client:
        r = await client.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
            json={"model": "llama-3.3-70b-versatile", "messages": [{"role": "user", "content": prompt}]},
            timeout=60
        )
        content = r.json()["choices"][0]["message"]["content"]
        start = content.find("{")
        end = content.rfind("}") + 1
        data = json.loads(content[start:end])
        items = data.get("facts") or data.get("tips") or data.get("mistakes") or data.get("items", [])

    print("🎨 Creating video segments...")
    segments = []

    # Hook
    hook_audio = temp_dir / "hook.mp3"
    await edge_tts.Communicate(format_config["hook"], "en-US-ChristopherNeural", rate="+10%").save(str(hook_audio))
    hook_img = temp_dir / "hook_bg.jpg"
    await get_vertical_image(topic, hook_img)

    audio = AudioFileClip(str(hook_audio))
    bg = ImageClip(str(hook_img)).with_duration(audio.duration).resized((1080, 1920))
    bg = bg.with_audio(audio)
    segments.append(bg)

    # Content items
    for i, item in enumerate(items[:5]):
        text = item if isinstance(item, str) else str(item)
        audio_path = temp_dir / f"audio_{i}.mp3"
        await edge_tts.Communicate(text, "en-US-ChristopherNeural", rate="+10%").save(str(audio_path))

        img_path = temp_dir / f"bg_{i}.jpg"
        await get_vertical_image(f"{topic} aesthetic", img_path)

        audio = AudioFileClip(str(audio_path))
        bg = ImageClip(str(img_path)).with_duration(audio.duration).resized((1080, 1920))
        bg = bg.with_audio(audio)
        segments.append(bg)
        print(f"   Segment {i+1}/5 complete")

    # CTA
    cta_audio = temp_dir / "cta.mp3"
    await edge_tts.Communicate(format_config["cta"], "en-US-ChristopherNeural", rate="+10%").save(str(cta_audio))
    audio = AudioFileClip(str(cta_audio))
    bg = ImageClip(str(hook_img)).with_duration(audio.duration).resized((1080, 1920))
    bg = bg.with_audio(audio)
    segments.append(bg)

    print("🔗 Combining segments...")
    final = concatenate_videoclips(segments, method="compose")

    output_path = str(output_dir / f"short_{topic.replace(' ', '_')[:20]}.mp4")
    final.write_videofile(output_path, fps=30, codec="libx264", audio_codec="aac")

    print(f"✅ Short saved: {output_path}")
    print(f"⏱️ Duration: {final.duration:.1f} seconds")
    return output_path

short_path = await generate_short(SHORT_TOPIC, FORMAT_TYPE)

In [ ]:
# Play the short
from IPython.display import Video
Video(short_path, embed=True, width=360)

In [ ]:
# Download the short
from google.colab import files
files.download(short_path)

## 4️⃣ Batch Generate Multiple Videos

In [ ]:
# Generate multiple videos at once
TOPICS = [
    "5 Facts About Black Holes",
    "How Bees Make Honey",
    "Secrets of Ancient Egypt"
]

for topic in TOPICS:
    print(f"\n{'='*50}")
    await generate_video(topic, 5)

print("\n✅ All videos generated!")

In [ ]:
# Download all videos as zip
!zip -r videos.zip outputs/
from google.colab import files
files.download('videos.zip')